# Notebook 02: Interceptor — Teleport Identity Injection

## Overview

The AgentCore Gateway validates the Teleport JWT cryptographically, but does not
automatically forward caller identity to Lambda targets. This notebook deploys a
REQUEST interceptor Lambda that runs before every tool call to bridge that gap:

```
tsh mcp connect  →  AgentCore Gateway (JWT validated)
                         ↓  passRequestHeaders: true
                 Interceptor Lambda
                   1. Reads Authorization: Bearer <teleport-jwt>
                   2. Decodes JWT payload  (no re-verify — gateway already did it)
                   3. Extracts sub + roles
                   4. Injects _teleport_user + _teleport_roles into tools/call arguments
                         ↓
                 Tool Lambda  ←  event now contains verified caller identity
```

After this notebook, `whoami_tool` returns the real Teleport identity instead of
the `unknown` placeholder.

### Prerequisites
- Notebook 01 completed (gateway and tool Lambda exist)
- `.env` populated with AWS credentials

## Step 1: Setup

In [ ]:
import os, json, time, io, zipfile, subprocess
import boto3
from dotenv import load_dotenv
from botocore.exceptions import ClientError

load_dotenv(dotenv_path='.env', override=True)
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
REGION = os.environ['AWS_DEFAULT_REGION']

# Explicit session forces boto3 to read from os.environ rather than its credential cache
session = boto3.Session(
    aws_access_key_id=os.environ.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.environ.get('AWS_SECRET_ACCESS_KEY'),
    aws_session_token=os.environ.get('AWS_SESSION_TOKEN'),
    region_name=REGION
)

iam            = session.client('iam')
lambda_client  = session.client('lambda')
gateway_client = session.client('bedrock-agentcore-control')
account_id     = session.client('sts').get_caller_identity()['Account']

# Constants from notebook 01
GATEWAY_NAME          = 'teleport-identity-demo'
TELEPORT_CLUSTER      = 'ellinj.teleport.sh'
TELEPORT_DISCOVERY    = f'https://{TELEPORT_CLUSTER}/.well-known/openid-configuration'
GATEWAY_ROLE_NAME     = 'teleport-demo-agentcore-gateway-role'

# Resolve gateway id + url
gw = next(g for g in gateway_client.list_gateways(maxResults=100)['items']
          if g['name'] == GATEWAY_NAME)
gateway_id       = gw['gatewayId']
gateway_url      = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['gatewayUrl']
gateway_role_arn = iam.get_role(RoleName=GATEWAY_ROLE_NAME)['Role']['Arn']

print(f'Account : {account_id}')
print(f'Gateway : {gateway_id}')
print(f'URL     : {gateway_url}')

## Step 2: Create IAM Role for Interceptor Lambda

In [ ]:
INTERCEPTOR_ROLE_NAME = 'teleport-demo-interceptor-lambda-role'

trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'lambda.amazonaws.com'},
        'Action': 'sts:AssumeRole'
    }]
}

try:
    resp = iam.create_role(
        RoleName=INTERCEPTOR_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Interceptor Lambda role for Teleport AgentCore demo'
    )
    interceptor_role_arn = resp['Role']['Arn']
    print(f'Created role: {interceptor_role_arn}')
    time.sleep(10)
except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        interceptor_role_arn = iam.get_role(RoleName=INTERCEPTOR_ROLE_NAME)['Role']['Arn']
        print(f'Role already exists: {interceptor_role_arn}')
    else:
        raise

iam.attach_role_policy(
    RoleName=INTERCEPTOR_ROLE_NAME,
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
)
print('Attached AWSLambdaBasicExecutionRole')

## Step 3: Deploy Interceptor Lambda

`lambda_interceptor.py` decodes the Teleport JWT from the `Authorization` header
and injects `_teleport_user` and `_teleport_roles` into `tools/call` arguments.

In [ ]:
INTERCEPTOR_LAMBDA_NAME = 'teleport-demo-interceptor'

with open('lambda_interceptor.py', 'r') as f:
    code = f.read()

buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', code)
buf.seek(0)
pkg = buf.read()
print(f'Packaged lambda_interceptor.py ({len(pkg):,} bytes)')

try:
    resp = lambda_client.create_function(
        FunctionName=INTERCEPTOR_LAMBDA_NAME,
        Runtime='python3.12',
        Role=interceptor_role_arn,
        Handler='lambda_function.lambda_handler',
        Code={'ZipFile': pkg},
        Description='Teleport JWT identity injector for AgentCore Gateway',
        Timeout=10,
        MemorySize=128,
        Tags={
            'teleport.dev/creator': 'none@nill.com',
            'demo': 'teleport-agentcore'
        }
    )
    interceptor_lambda_arn = resp['FunctionArn']
    print(f'Created: {interceptor_lambda_arn}')
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceConflictException':
        resp = lambda_client.update_function_code(
            FunctionName=INTERCEPTOR_LAMBDA_NAME, ZipFile=pkg
        )
        interceptor_lambda_arn = resp['FunctionArn']
        print(f'Updated: {interceptor_lambda_arn}')
    else:
        raise

lambda_client.get_waiter('function_active_v2').wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)
print('Lambda is active')

## Step 4: Grant Gateway Permission to Invoke Interceptor

In [ ]:
try:
    lambda_client.add_permission(
        FunctionName=INTERCEPTOR_LAMBDA_NAME,
        StatementId='AllowGatewayInvoke',
        Action='lambda:InvokeFunction',
        Principal='bedrock-agentcore.amazonaws.com',
        SourceArn=f'arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/*'
    )
    print('Permission added')
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceConflictException':
        print('Permission already exists')
    else:
        raise

# Also grant the gateway IAM role permission to invoke the interceptor
iam.put_role_policy(
    RoleName=GATEWAY_ROLE_NAME,
    PolicyName='InterceptorInvokePolicy',
    PolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [{
            'Effect': 'Allow',
            'Action': 'lambda:InvokeFunction',
            'Resource': interceptor_lambda_arn
        }]
    })
)
print('Gateway role updated with interceptor invoke permission')

## Step 5: Wire Interceptor into the Gateway

Update the gateway to run the interceptor on every `REQUEST` with `passRequestHeaders: true`
so the `Authorization: Bearer <teleport-jwt>` header is forwarded to the interceptor.

In [ ]:

# The gateway URL from the API omits the /mcp suffix, but Teleport appends /mcp
# in teleport.yaml and sets that as the JWT aud claim. Add it explicitly.
mcp_audience = f'mcp+{gateway_url.rstrip("/")}/mcp'
print(f'Setting audience: {mcp_audience}')

gateway_client.update_gateway(
    gatewayIdentifier=gateway_id,
    name=GATEWAY_NAME,
    roleArn=gateway_role_arn,
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration={
        'customJWTAuthorizer': {
            'discoveryUrl': TELEPORT_DISCOVERY,
            'allowedAudience': [mcp_audience],
            'customClaims': [{
                'inboundTokenClaimName': 'roles',
                'inboundTokenClaimValueType': 'STRING_ARRAY',
                'authorizingClaimMatchValue': {
                    'claimMatchValue': {'matchValueString': 'mcp-user'},
                    'claimMatchOperator': 'CONTAINS'
                }
            }]
        }
    },
    interceptorConfigurations=[{
        'interceptor': {'lambda': {'arn': interceptor_lambda_arn}},
        'interceptionPoints': ['REQUEST'],
        'inputConfiguration': {'passRequestHeaders': True}
    }]
)
print('Gateway updated with interceptor')
print(f'  Interceptor : {interceptor_lambda_arn}')
print(f'  Headers     : passRequestHeaders=True')

## Step 6: Wait for Gateway to be READY

In [ ]:
print('Waiting for gateway to reach READY status...')
while True:
    status = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['status']
    print(f'  Status: {status}')
    if status == 'READY':
        break
    if status == 'FAILED':
        raise RuntimeError('Gateway update failed')
    time.sleep(10)
print('Gateway is READY')

## Step 7: Test — whoami_tool Should Now Return Real Identity

Call `whoami_tool` via `tsh mcp connect`. The interceptor decodes the Teleport JWT
and injects the identity before the tool Lambda runs.

In [ ]:
import json as _json

APP_NAME = 'agentcore-gateway'  # name in teleport.yaml

msgs = [
    '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"notebook","version":"0.0.1"}}}',
    '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"TeleportDemo___whoami_tool","arguments":{}}}',
]

result = subprocess.run(
    ['tsh', 'mcp', 'connect', APP_NAME],
    input='\n'.join(msgs) + '\n',
    capture_output=True, text=True, timeout=30
)

for line in result.stdout.splitlines():
    try:
        msg = _json.loads(line)
        if msg.get('id') == 2:
            text = msg['result']['content'][0]['text']
            outer = _json.loads(text)
            identity = _json.loads(outer['body'])
            print('✓ whoami_tool — verified Teleport identity:')
            print(_json.dumps(identity, indent=2))
    except Exception:
        pass

# tsh emits "server does not support listening" at INFO level on session close — ignore it
real_errors = [l for l in result.stderr.splitlines()
               if '"level":"error"' in l or ('ERROR' in l and 'listening' not in l)]
for e in real_errors:
    print('ERROR:', e)

## Step 8: Test — get_order_tool Includes Caller Identity

In [ ]:
msgs = [
    '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"notebook","version":"0.0.1"}}}',
    '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"TeleportDemo___get_order_tool","arguments":{"orderId":"ORD-42"}}}',
]

result = subprocess.run(
    ['tsh', 'mcp', 'connect', APP_NAME],
    input='\n'.join(msgs) + '\n',
    capture_output=True, text=True, timeout=30
)

for line in result.stdout.splitlines():
    try:
        parsed = _json.loads(line)
        if parsed.get('id') == 2:
            print('get_order_tool response:')
            content = parsed.get('result', {}).get('content', [])
            for item in content:
                body = _json.loads(_json.loads(item['text'])['body'])
                print(_json.dumps(body, indent=2))
    except Exception:
        pass

## What's Next

Identity now flows from Teleport through the gateway into every tool call.
Notebook 03 adds Cedar policy enforcement via Amazon Verified Permissions —
the interceptor will call AVP before forwarding, and `update_order_tool` will
be blocked for non-admin Teleport roles without any code change to the tool Lambda.

## Cleanup (Interceptor Only)

To remove just the interceptor and revert the gateway to its notebook-01 state:

In [ ]:
from botocore.exceptions import ClientError

# The API rejects interceptorConfigurations=[] (min length 1), so we can't remove
# the interceptor config here. Notebook 01 cleanup deletes the gateway entirely.
# This cell just removes the Lambda and IAM role.

def _try(fn, *args, codes=('NoSuchEntity', 'ResourceNotFoundException', 'ResourceConflictException'), **kwargs):
    try:
        return fn(*args, **kwargs)
    except ClientError as e:
        if e.response['Error']['Code'] in codes:
            print(f'  Already gone: {e.response["Error"]["Code"]}')
        else:
            raise

# Remove interceptor invoke permission from gateway role
_try(iam.delete_role_policy, RoleName=GATEWAY_ROLE_NAME, PolicyName='InterceptorInvokePolicy')
print('Removed InterceptorInvokePolicy from gateway role')

# Delete Lambda
_try(lambda_client.delete_function, FunctionName=INTERCEPTOR_LAMBDA_NAME)
print(f'Deleted Lambda: {INTERCEPTOR_LAMBDA_NAME}')

# Delete interceptor IAM role
for p in iam.list_attached_role_policies(RoleName=INTERCEPTOR_ROLE_NAME).get('AttachedPolicies', []):
    _try(iam.detach_role_policy, RoleName=INTERCEPTOR_ROLE_NAME, PolicyArn=p['PolicyArn'])
for p in iam.list_role_policies(RoleName=INTERCEPTOR_ROLE_NAME).get('PolicyNames', []):
    _try(iam.delete_role_policy, RoleName=INTERCEPTOR_ROLE_NAME, PolicyName=p)
_try(iam.delete_role, RoleName=INTERCEPTOR_ROLE_NAME)
print(f'Deleted role: {INTERCEPTOR_ROLE_NAME}')

print()
print('Run notebook 01 cleanup to delete the gateway itself.')